In [6]:
import os
import struct
import pandas as pd
pd.set_option('display.max_columns',150)
import warnings
warnings.filterwarnings('ignore')
import re

### To-Do

* 마지막이 아닌데, ncode 10으로 되어있는 행 제거
* column 명 맞추기
* raw_data, stepEnd 따로 빼기
* config.py에 apro pne 따로 분리해서 추가하기

### Done

* binary_data 읽기
* pandas dataframe으로 변경하기
* raw data 전처리하기
* step end 추출하기
* raw data, step end 컬럼명 바꿔서 저장하기


In [7]:
# json 으로 만들기
col_dict = {
'nStep':'Step',
'nStatus':'Type',
'nCode':'상태',
'TIME[HMS]':'TIME[HMS]', 
'nVol':'전압[V]', 
'nCur':'전류[A]',
'nTemp':'온도(℃)',
'nCap':'용량[Ah]', 
'C_cap':'충전용량(mAh)', 
'D_cap':'방전용량(mAh)', 
'nWatt':'Watt[W]', 
'nWH':'WattHour[Wh]',
'C_work':'충전WattHour[Wh]', 
'D_work':'방전WattHour[Wh]', 
'nAvgVol':'평균전압[V]',
'nAvgCur':'평균전류[A]', 
'nDCIR':'Imp[mΩ]',
'총시간[HMS]':'총시간[HMS]',
'nCycle':'총Cycle'
}
# 'index':'인덱스', 
# 'cum_cap':'총 용량(mAh)', 
# 'cum_work':'총 WattHour(mWh)', 
# 'nCVMode':'충전CV모드', 
# 'CVCapacity':'충전CV 용량(mAh)', 

# temp['nStatus']
# {4:'충전', 5:'',  8:'방전', 11:'', 28:'Rest'} #5 충전, 11방전인데 겹치네.. 뭔지 차이는 찾아봐야 함.

In [8]:
col_dict = \
{'nStatus':'StepType',
 'nCode':'Code',
 'nFault':'Fault', 
 'nCycle':'TotalCycle', 
 'nStep':'StepNo', 
 'nTemp':'Temp',
 'nCur':'Current', 
 'nVol':'Voltage',
 'nCap':'Capacity', 
 'nWatt':'Power', 
 'nWH':'WattHour', 
 'nTime':'TotalTime_sec', 
 'index':'DataSeq', 
 'nCurCycle':'Cyclenum', 
 'nDCIR':'Impedance',
 'cum_cap':'IntegralCapacity', 
 'C_cap':'ChargeCapacity', 
 'D_cap':'DischargeCapacity', 
 'cum_work':'IntergralWattHour', 
 'C_work':'ChargeWattHour', 
 'D_work':'DischargeWattHour',
 'nAvgVol':'AvgVoltage', 
 'nAvgCur':'AvgCurrent', 
 'nCVMode':'CVMode', 
 'CVRunTime':'CVEndTime', #맞나?? 
 'CVCapacity':'ChargeCVCap',
 'nWorkingTime':'WorkingTime'}

### Read  and convert to Dataframe

In [9]:
def _parse_year(file_path):
    file_path = os.path.basename(file_path)
    # 정규표현식: 언더바 사이의 숫자 6~8자리를 찾고 연도 부분만 추출
    pattern = re.compile(r'_(\d{2,4})\d{4}_')

    match = pattern.search(file_path)
    if match:
        year_full = match.group(1) # '24' 또는 '2024'
        year_2digit = year_full[-2:] # 뒤에서 두 글자만 슬라이싱
        return int(year_2digit)
            
def read_binary(file_path):
    record_list = []
    year = _parse_year(file_path)
    format_string_1010 = ( # file_format_1010
        "@" # for little-endian byte order
        "BBB"  # nStatus, nCode, nFault (BYTE 3개)
        "hhh"   # nCycle, nStep, nTemp (short 3개)
        "iiiii"  # nCur, nVol, nCap, nWatt, nWH (int 5개)
        "II"   # nTime, index (UINT 2개)
        "h"    # nCurCycle (short 1개)
        "iiiiiiiii"    # nDCIR, cum_cap ,C_cap, D_cap, cum_work, C_work, D_work, nAvgVol, nAvgCur (int 9개)
        "B" # nCVMode (unsigned char 1개)
        "III" # CVRunTime, CVCapacity, nWorkingTime (UNIT 3개)
    )
    
    format_string_0912 = ( # file_format_0912
        "@" # for little-endian byte order
        "BBB"  # nStatus, nCode, nFault (BYTE 3개)
        "hhh"   # nCycle, nStep, nTemp (short 3개)
        "iiiii"  # nCur, nVol, nCap, nWatt, nWH (int 5개)
        "II"   # nTime, index (UINT 2개)
        "h"    # nCurCycle (short 1개)
        "iiiiiiiii"    # nDCIR, cum_cap ,C_cap, D_cap, cum_work, C_work, D_work, nAvgVol, nAvgCur (int 9개)
        "B" # nCVMode (unsigned char 1개)
        "II" # CVCapacity, nWorkingTime (UNIT 3개)
    )
    
    format_string_0932 = ( # file_format_0912
        "@" # for little-endian byte order
        "BBB"  # nStatus, nCode, nFault (BYTE 3개)
        "hhh"   # nCycle, nStep, nTemp (short 3개)
        "iiiii"  # nCur, nVol, nCap, nWatt, nWH (int 5개)
        "II"   # nTime, index (UINT 2개)
        "h"    # nCurCycle (short 1개)
        "iiiiiiiii"    # nDCIR, cum_cap ,C_cap, D_cap, cum_work, C_work, D_work, nAvgVol, nAvgCur (int 9개)
    )
    # 포맷의 크기 계산
    
        
    with open(file_path, 'rb') as file:
        file_size = os.fstat(file.fileno()).st_size
        print(file_size, year)
        if year != None:
            if year == 22 and file_size % 80 == 0:
                format_string = format_string_0932
                print("0932")
            elif year == 23 and file_size % 92 == 0:
                format_string = format_string_0912
                print("0912")
            elif year >= 24 and file_size % 96 == 0:   
                format_string = format_string_1010
                print("1010")   
            else:
                if file_size % 80 == 0:
                    format_string = format_string_0932
                    print("0932")
                elif file_size % 92 == 0:
                    format_string = format_string_0912
                    print("0912")
                elif file_size % 96 == 0:   
                    format_string = format_string_1010
                    print("1010")   
        else:
            if file_size % 96 == 0:
                format_string = format_string_1010
                print("1010")
            elif file_size % 92 == 0:
                format_string = format_string_0912
                print("0912")
            elif file_size % 80 == 0:   
                format_string = format_string_0932
                print("0932")   
            else:
                format_string = format_string_1010
                print("1010")
        
        record_size = struct.calcsize(format_string)
        
        while True:
            data = file.read(record_size)
            if not data:
                break
            record = struct.unpack(format_string, data)
            record_list.append(record)
            
    return record_list

def convert_binary_to_df(record_list):
    if len(record_list) == 0:
        return pd.DataFrame([], columns=[])
    
    if len(record_list[0]) == 27:
        col = ['nStatus','nCode', 'nFault', 'nCycle', 'nStep', 'nTemp', 'nCur', 'nVol',
        'nCap', 'nWatt', 'nWH', 'nTime', 'index', 'nCurCycle', 'nDCIR',
        'cum_cap', 'C_cap', 'D_cap', 'cum_work', 'C_work', 'D_work',
        'nAvgVol', 'nAvgCur', 'nCVMode', 'CVRunTime', 'CVCapacity','nWorkingTime']
    elif len(record_list[0]) == 26:
        col = ['nStatus','nCode', 'nFault', 'nCycle', 'nStep', 'nTemp', 'nCur', 'nVol',
        'nCap', 'nWatt', 'nWH', 'nTime', 'index', 'nCurCycle', 'nDCIR',
        'cum_cap', 'C_cap', 'D_cap', 'cum_work', 'C_work', 'D_work',
        'nAvgVol', 'nAvgCur', 'nCVMode', 'CVCapacity','nWorkingTime']
    elif len(record_list[0]) == 23:
        col = ['nStatus','nCode', 'nFault', 'nCycle', 'nStep', 'nTemp', 'nCur', 'nVol',
        'nCap', 'nWatt', 'nWH', 'nTime', 'index', 'nCurCycle', 'nDCIR',
        'cum_cap', 'C_cap', 'D_cap', 'cum_work', 'C_work', 'D_work',
        'nAvgVol', 'nAvgCur']
    raw_data = pd.DataFrame(record_list, columns=col)
    
    return raw_data

### Raw data함수

In [10]:
def del_abnormal(df):
    # ncode 10이면서 마지막행 아닌것 제거하기
    drop_idx = df[df['nCode'] == 10].index[:-1]
    df = df.drop(drop_idx)
    return df

def convert_time(value):
    return pd.to_datetime(value * 10**6).dt.time

def convert_nTime(df):
    # "공정 시간"을 초 단위로 timedelta 변환 (100으로 나눠서 초로 변환)
    df['TotalTime_diff'] = df['TotalTime_sec'].diff().fillna(0)
    df['TotalTime'] = pd.to_timedelta(df['TotalTime_sec'], unit='s')
    # 누적 시간 → "D:HH:MM:SS" 변환
    df['TotalTime'] = df['TotalTime'].apply(lambda x: f"{x.days}:{x.components.hours:02}:{x.components.minutes:02}:{x.components.seconds:02}")
    # 개별 공정 시간 계산 (현재 행 - 이전 행, 첫 번째 행은 그대로 사용)
    df['StepTime_sec'] = df.groupby('group_id')['TotalTime_diff'].cumsum().astype('float')
    df['StepTime']  = pd.to_timedelta(df['StepTime_sec'] , unit='s')
    # 개별 공정 시간 → "HH:MM:SS" 변환
    df['StepTime'] = df['StepTime'].apply(lambda x: f"{x.components.hours:02}:{x.components.minutes:02}:{x.components.seconds:02}")
    return df

def convert_ncode(value):
    ncode_dict = {0:'Normal', 1:"Time Complete", 2:"Voltage Complete", 3:"Current Complete", 4:"Capacity Complete", 10:"Stopped by User"}
    return ncode_dict.get(value, "None")

def convert_type(value):
    ncode_dict = {2:'Charge', 4:'Charge', 5:'Charge', 8:'Discharge', 11:'Discharge', 17:'Pattern', 28:'Rest'} 
    return ncode_dict.get(value, "None")

def apply_fraction(df):
    df['nTemp'] = df['nTemp'] * 0.1
    df['nTime'] = df['nTime'] * 0.01
    df[['nCur', 'nVol', 'nAvgVol', 'nAvgCur']] = df[['nCur', 'nVol', 'nAvgVol', 'nAvgCur']] * 0.0001
    df[['nCap','nWatt','nWH','nDCIR','cum_cap','C_cap','D_cap','cum_work','C_work','D_work']] = df[['nCap','nWatt','nWH','nDCIR','cum_cap','C_cap','D_cap','cum_work','C_work','D_work']]*0.001
    return df

def process_rpt(stepend, cell_type):
    if cell_type == 'JF1':
        criteria_current = 14.5
    elif cell_type == 'JF1R':
        criteria_current = 8.625
    elif cell_type == 'JF2':
        criteria_current = 39.80
    elif cell_type == 'JF2S':
        criteria_current = 38.950
    elif cell_type == 'JF3':
        criteria_current = 43.275
    else:
        criteria_current = None
        
    stepend['TestMode'] = 'In-situ'
    stepend['IsRptCapa'] = False
    stepend['IsRptDcir'] = False
    
    temp = stepend['StepNo'].value_counts()
    rpt_temp = temp[temp < temp.values.mean()].sort_index()

    stepend_rpt = stepend[(stepend['StepNo'].isin(rpt_temp.index))]
    stepend.loc[stepend_rpt.index, 'TestMode'] = 'RPT'

    def cluster_numbers_by_difference(numbers, threshold=10):
        if not numbers:
            return []

        clusters = []
        current_cluster = [numbers[0]]

        for i in range(1, len(numbers)):
            if numbers[i] - numbers[i-1] > threshold:
                clusters.append(current_cluster)
                current_cluster = [numbers[i]]
            else:
                current_cluster.append(numbers[i])

        clusters.append(current_cluster)

        return clusters

    rpt_group = cluster_numbers_by_difference(stepend_rpt.index.tolist())  #rpt 별로 group 을 만들어준다. 그룹 내에서 capa가 제일 큰걸로 rpt 용량인지 판별하기 때문에
    rpt_capa_idx, rpt_dcir_idx = [], []
    
    for mini_group in rpt_group:
        rpt_df = stepend.loc[mini_group]
        rpt_df = rpt_df[(rpt_df['StepType'] == 'Discharge') | (rpt_df['StepType'] == 'Charge')]
        if rpt_df.empty:
            continue
        
        if criteria_current is not None:
            print(criteria_current)
            condition = (rpt_df["Current"].abs() >= criteria_current - 2) & \
                    (rpt_df["Current"].abs() <= criteria_current + 2) & (rpt_df['StepType'] == 'Discharge')

            rpt_idx_discharge_df = rpt_df[condition]
            print(rpt_idx_discharge_df)
            if len(rpt_idx_discharge_df) != 0:
                if len(rpt_idx_discharge_df) >= 2:
                    rpt_idx_discharge = rpt_idx_discharge_df["DischargeCapacity"].idxmax()
                else:
                    rpt_idx_discharge = rpt_idx_discharge_df.index[0]
            else:
                rpt_idx_discharge = rpt_df["DischargeCapacity"].idxmax() 
            
        else:
            rpt_idx_discharge = rpt_df["DischargeCapacity"].idxmax() 

        charge_df = rpt_df.loc[:rpt_idx_discharge].query("StepType == 'Charge'")
        rpt_idx_charge = charge_df.index.max() if not charge_df.empty else None
        
        dcir_df = rpt_df.loc[rpt_idx_discharge+1:].query("StepType == 'Discharge'")
        dcir_idx = dcir_df.index.min() if not dcir_df.empty else None
        print(f"rpt_idx_charge : {rpt_idx_charge} / rpt_idx_discharge : {rpt_idx_discharge} / dcir_idx : {dcir_idx}")
        print(rpt_df[['StepType', 'Current', 'Voltage', 'ChargeCapacity', 'DischargeCapacity']])
        
        rpt_capa_idx.extend([rpt_idx_charge, rpt_idx_discharge])
        rpt_dcir_idx.append(dcir_idx)
        
    stepend.loc[rpt_capa_idx, 'IsRptCapa'] = True
    stepend.loc[(stepend["StepType"] == "Discharge") & (stepend["Code"] == "Time Complete"), "IsRptDcir"] = True
    
    return stepend 

def preprocess(df):
    df = del_abnormal(df)
    df = apply_fraction(df)
    df = df.rename(columns=col_dict) 
    df['Code'] = df['Code'].apply(convert_ncode)
    df['StepType'] = df['StepType'].apply(convert_type)
    df['group_id'] = (df['StepType'] != df['StepType'].shift(1)).cumsum()
    df = convert_nTime(df)
    
    stepend_df = df.query("Code!='Normal'").reset_index(drop=True)
    # stepend_df = process_rpt(stepend_df)
    
    return df, stepend_df


In [11]:
file_path = r'C:\Users\jhchoei\Desktop\Workspace\VoltaFlow\data\error 확인용\20241226_JF2_45oC_0.5CP 0.67CP_Cycle_UDTF10123\M05CH018[018]\20241226_JF2_45oC_0.5CP 0.67CP_Cycle_UDTF10123.dat'
binary_data = read_binary(file_path)
raw_data = convert_binary_to_df(binary_data)
df, stepend_data = preprocess(raw_data)

31480704 None
1010


In [12]:
stepend_data

,StepType,Code,Fault,TotalCycle,StepNo,Temp,Current,Voltage,Capacity,Power,WattHour,TotalTime_sec,DataSeq,Cyclenum,Impedance,IntegralCapacity,ChargeCapacity,DischargeCapacity,IntergralWattHour,ChargeWattHour,DischargeWattHour,AvgVoltage,AvgCurrent,CVMode,CVEndTime,ChargeCVCap,WorkingTime,group_id,TotalTime_diff,TotalTime,StepTime_sec,StepTime
0,Rest,Time Complete,0,1,2,45.2,0.0,3.2372,0.0,0.0,0.0,14400.00,14400,1,0.0,0.000,0.0,0.0,0.000,0.0,0.0,3.2373,0.0,0,0,0,14407,1,1.0,0:04:00:00,14400.0,04:00:00
1,None,None,32,2,5,45.2,0.0,3.2372,0.0,0.0,0.0,14400.00,14401,2,0.0,0.000,0.0,0.0,0.000,0.0,0.0,3.2372,0.0,0,0,0,14408,2,0.0,0:04:00:00,0.0,00:00:00
2,None,None,32,2,5,45.2,0.0,3.2372,0.0,0.0,0.0,14400.00,14402,2,0.0,0.000,0.0,0.0,0.000,0.0,0.0,3.2372,0.0,0,0,0,14409,2,0.0,0:04:00:00,0.0,00:00:00
3,None,None,32,2,5,45.2,0.0,3.2372,0.0,0.0,0.0,14400.00,14403,2,0.0,0.000,0.0,0.0,0.000,0.0,0.0,3.2372,0.0,0,0,0,14410,2,0.0,0:04:00:00,0.0,00:00:00
4,None,None,32,2,5,45.2,0.0,3.2372,0.0,0.0,0.0,14400.00,14404,2,0.0,0.000,0.0,0.0,0.000,0.0,0.0,3.2372,0.0,0,0,0,14411,2,0.0,0:04:00:00,0.0,00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287278,None,None,32,2,9,45.3,0.0,3.3710,0.0,0.0,0.0,40639.52,327919,2,0.0,138.047,0.0,0.0,467.529,0.0,0.0,3.3870,0.0,0,0,0,327945,7,0.0,0:11:17:19,0.0,00:00:00
287279,None,None,32,2,9,45.3,0.0,3.3710,0.0,0.0,0.0,40639.52,327920,2,0.0,138.047,0.0,0.0,467.529,0.0,0.0,3.3870,0.0,0,0,0,327946,7,0.0,0:11:17:19,0.0,00:00:00
287280,None,None,32,2,9,45.3,0.0,3.3710,0.0,0.0,0.0,40639.52,327921,2,0.0,138.047,0.0,0.0,467.529,0.0,0.0,3.3870,0.0,0,0,0,327947,7,0.0,0:11:17:19,0.0,00:00:00
287281,None,None,32,2,9,45.3,0.0,3.3710,0.0,0.0,0.0,40639.52,327922,2,0.0,138.047,0.0,0.0,467.529,0.0,0.0,3.3870,0.0,0,0,0,327948,7,0.0,0:11:17:19,0.0,00:00:00


In [29]:
stepend_data["Cell_ID"] = "UDKTF10021"
stepend_data["EXP_ID"] = "EXP_250511_00097"

In [1]:
import psycopg2
from io import StringIO

try:
    conn = psycopg2.connect(host="10.99.212.29",
                            database="mart_db",
                            user="postgres",
                            password="lifemodeling",
                            port=5432,
                            client_encoding='utf-8') # 이 부분을 추가
    print("데이터베이스 연결 성공!")
    # 여기에 데이터베이스 작업 코드 추가
    cur = conn.cursor()
    csv_buffer = StringIO()
    temp = stepend_data[['Cell_ID', 'EXP_ID', 'StepNo', 'StepType', 'Code','TotalCycle', 'Voltage', 'Current',
                        'Capacity', 'Power', 'WattHour', 'StepTime', 'TotalTime', 'Impedance', 'Temp',
                        'AvgVoltage', 'AvgCurrent', 'ChargeCapacity', 'DischargeCapacity', 'ChargeWattHour',
                        'DischargeWattHour', 'CVEndTime', 'TestMode', 'IsRptCapa', 'IsRptDcir']]
    temp.to_csv(csv_buffer, index=False, header=False, sep='\t', encoding='utf-8')
    csv_buffer.seek(0)  # StringIO 버퍼의 시작으로 이동
    cur.copy_from(csv_buffer, 'exp_fact_tb', sep='\t', columns=['cell_id', 'exp_id', 'stepno', 'steptype', 'code', 'totalcycle', 'voltage', 'current', 'capacity', 'power', 'watthour', 'steptime', 'totaltime', 'impedance', 'temp', 'avgvoltage', 'avgcurrent', 'chargecapacity', 'dischargecapacity', 'chargewatthour', 'dischargewatthour', 'cvendtime', 'testmode', 'isrptcapa', 'isrptdcir'])
    conn.commit()  # 변경 사항을 커밋합니다.
    # cur.execute("SELECT * FROM exp_fact_tb LIMIT 0")  # 테이블 구조 확인
    # # --- 여기서부터 컬럼명을 확인하는 코드 ---
    # if cur.description:
    #     column_names = [col[0] for col in cur.description]
    #     print("테이블 컬럼명:", column_names)
    # else:
    #     print("컬럼 정보를 가져올 수 없습니다. 쿼리가 올바른지 확인해주세요.")
    # --- 여기까지 컬럼명을 확인하는 코드 ---
    #cur.execute("INSERT INTO exp_fact_tb(CELL_ID, EXP_ID) VALUES ('ASD', 'SBS')")
    #conn.commit()

except Exception as e:
    print(f"데이터베이스 오류 발생: {e}")

finally:
    if cur:
        cur.close()
    if conn:
        conn.close()

데이터베이스 연결 성공!
데이터베이스 오류 발생: name 'stepend_data' is not defined


In [130]:
stepend_data

,StepType,Code,Fault,TotalCycle,StepNo,Temp,Current,Voltage,Capacity,Power,WattHour,TotalTime_sec,DataSeq,Cyclenum,Impedance,IntegralCapacity,ChargeCapacity,DischargeCapacity,IntergralWattHour,ChargeWattHour,DischargeWattHour,AvgVoltage,AvgCurrent,CVMode,CVEndTime,ChargeCVCap,WorkingTime,group_id,TotalTime_diff,TotalTime,StepTime_sec,StepTime,TestMode,IsRptCapa,IsRptDcir,Cell_ID,EXP_ID
0,Rest,Time Complete,0,1,2,23.8,0.0000,2.8020,0.000,0.000,0.000,14400.00,240,1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.8006,0.0000,0,0,0,14406,1,60.00,0:04:00:00,14400.00,04:00:00,In-situ,False,False,UDKTF10021,EXP_250511_00097
1,Discharge,Voltage Complete,0,2,5,23.2,-38.9359,2.4999,1.042,-97.335,2.759,14496.38,243,2,1.576,-1.042,0.000,1.042,-2.759,0.000,2.759,2.6464,-38.9377,0,0,0,14503,2,36.38,0:04:01:36,96.38,00:01:36,In-situ,False,False,UDKTF10021,EXP_250511_00097
2,Rest,Time Complete,0,2,6,23.2,0.0000,2.6047,0.000,0.000,0.000,16296.38,274,2,0.000,-1.042,0.000,0.000,-2.759,0.000,0.000,2.5993,0.0000,0,0,0,16303,3,60.00,0:04:31:36,1800.00,00:30:00,In-situ,False,False,UDKTF10021,EXP_250511_00097
3,Charge,Current Complete,0,3,9,24.7,7.7920,3.6500,151.357,28.440,507.590,30383.21,510,3,2.318,151.357,151.357,0.000,507.590,507.590,0.000,3.3557,38.6778,1,182000,862,30390,4,46.83,0:08:26:23,14086.83,03:54:46,In-situ,False,False,UDKTF10021,EXP_250511_00097
4,Rest,Time Complete,0,3,10,24.1,0.0000,3.4637,0.000,0.000,0.000,31583.21,531,3,0.000,151.357,0.000,0.000,507.590,0.000,0.000,3.4885,0.0000,0,0,0,31590,5,60.00,0:08:46:23,1200.00,00:20:00,In-situ,False,False,UDKTF10021,EXP_250511_00097
5,Discharge,Voltage Complete,0,3,11,24.6,-38.9355,2.5000,150.591,-97.338,482.996,45506.84,765,3,1.549,0.766,0.000,150.591,24.594,0.000,482.996,3.2073,-38.9374,0,0,0,45514,6,3.63,0:12:38:26,13923.63,03:52:03,In-situ,False,False,UDKTF10021,EXP_250511_00097
6,Rest,Time Complete,0,3,12,23.6,0.0000,2.7259,0.000,0.000,0.000,47306.84,796,3,0.000,0.766,0.000,0.000,24.594,0.000,0.000,2.6804,0.0000,0,0,0,47314,7,60.00,0:13:08:26,1800.00,00:30:00,In-situ,False,False,UDKTF10021,EXP_250511_00097
7,Charge,Current Complete,0,4,15,24.2,7.7920,3.6500,150.652,28.440,504.925,61280.57,1030,4,1.625,150.652,150.652,0.000,504.925,504.925,0.000,3.3527,38.8089,1,94000,428,61288,8,53.73,0:17:01:20,13973.73,03:52:53,In-situ,False,False,UDKTF10021,EXP_250511_00097
8,Rest,Time Complete,0,4,16,24.2,0.0000,3.5018,0.000,0.000,0.000,62480.57,1051,4,0.000,150.652,0.000,0.000,504.925,0.000,0.000,3.5165,0.0000,0,0,0,62488,9,60.00,0:17:21:20,1200.00,00:20:00,In-situ,False,False,UDKTF10021,EXP_250511_00097
9,Discharge,Time Complete,0,4,17,24.5,-77.8867,3.3703,0.216,-262.501,0.735,62490.57,1152,4,1.688,150.436,0.000,0.216,504.190,0.000,0.735,3.3995,-77.8879,0,0,0,62499,10,0.10,0:17:21:30,10.00,00:00:10,In-situ,False,False,UDKTF10021,EXP_250511_00097


In [88]:
stepend_data["TotalTime"]

0       0:04:00:00
1       0:05:37:03
2       0:06:07:03
3       0:10:26:02
4       0:10:46:02
          ...     
797    46:15:11:06
798    46:16:11:06
799    46:18:19:12
800    46:18:39:12
801    46:20:17:55
Name: TotalTime, Length: 802, dtype: object

In [14]:
set(stepend_data.columns).intersection(set(['No', 'StepNo', 'State', 'StepType', 'DataSelect', 'Code', 'IndexFrom',
       'IndexTo', 'CurrentCycleNum', 'TotalCycle', 'SaveSequence', 'Voltage',
       'Current', 'Capacity', 'Power', 'WattHour', 'StepTime', 'TotalTime',
       'Impedance', 'Temp', 'AvgVoltage', 'AvgCurrent', 'ChargeCapacity',
       'DischargeCapacity', 'ChargeWattHour', 'DischargeWattHour', 'CVEndTime',
       'TotalTimeCarry', 'Reserved3', 'CycleNum', 'ChamberTemp',
       'ChargeCCCapacity', 'ChargeCVCapacity', 'DischargeCCCapacity', 'ChCode',
       'Impedance100ms', 'Impedance1s', 'Impedance5s', 'Impedance30s',
       'Impedance60s', 'RealDate', 'RealClock', 'TestMode', 'IsRptCapa',
       'IsRptDcir']))

{'AvgCurrent',
 'AvgVoltage',
 'CVEndTime',
 'Capacity',
 'ChargeCapacity',
 'ChargeWattHour',
 'Code',
 'Current',
 'DischargeCapacity',
 'DischargeWattHour',
 'Impedance',
 'IsRptCapa',
 'IsRptDcir',
 'Power',
 'StepNo',
 'StepTime',
 'StepType',
 'Temp',
 'TestMode',
 'TotalCycle',
 'TotalTime',
 'Voltage',
 'WattHour'}

In [ ]:
import sys
sys.path.append(r"C:\Users\jhchoei\Desktop\Workspace\TOS")
from processor.APRO import AproProcessor
import config

processor = AproProcessor(config)
processor.parse_binary_file(file_path)

In [2]:
file_path = r'C:\Users\jhchoei\Desktop\Workspace\TOS\data\07004515_20250304_JF2S_CV_ACC_55_105CP_1CP_UDKTF10206\M07CH028[028]\07004515_20250304_JF2S_CV_ACC_55_105CP_1CP_UDKTF10206.dat'

In [ ]:
processor = AproProcessor(config)
processor.parse_binary_file(file_path)

In [5]:
processor.parse_binary_file(file_path)['StepEndData']

,StepType,Code,Fault,TotalCycle,StepNo,Temp,Current,Voltage,Capacity,Power,...,AvgCurrent,CVMode,CVEndTime,ChargeCVCap,WorkingTime,group_id,TotalTime_diff,TotalTime,StepTime_sec,StepTime
0,Rest,Time Complete,0,1,2,57.4,0.0000,3.2532,0.000,0.000,...,0.0000,0,0,0,14406,1,60.00,0:04:00:00,14400.00,04:00:00
1,Discharge,Voltage Complete,0,2,5,58.3,-38.9364,2.5000,34.996,-97.341,...,-38.9375,0,0,0,17642,2,55.67,0:04:53:55,3235.67,00:53:55
2,Rest,Time Complete,0,2,6,58.0,0.0000,2.6992,0.000,0.000,...,0.0000,0,0,0,19442,3,60.00,0:05:23:55,1800.00,00:30:00
3,Charge,Current Complete,0,3,9,58.3,7.7920,3.6500,163.177,28.440,...,38.7888,1,105000,410,34586,4,23.86,0:09:36:19,15143.86,04:12:23
4,Rest,Time Complete,0,3,10,58.0,0.0000,3.4733,0.000,0.000,...,0.0000,0,0,0,35786,5,60.00,0:09:56:19,1200.00,00:20:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,Rest,Time Complete,0,34,29,59.8,0.0000,2.8025,0.000,0.000,...,0.0000,0,0,0,529556,135,60.00,6:03:05:00,3600.00,01:00:00
165,Charge,Voltage Complete,0,35,25,59.8,148.3483,3.3610,34.613,498.598,...,152.3211,0,0,0,530375,136,37.96,6:03:18:38,817.96,00:13:37
166,Charge,Voltage Complete,0,35,26,60.0,68.2993,3.6501,124.796,249.299,...,73.7444,0,0,0,536467,136,32.01,6:05:00:10,6909.97,01:55:09
167,Rest,Time Complete,0,35,27,59.2,0.0000,3.5317,0.000,0.000,...,0.0000,0,0,0,537667,137,60.00,6:05:20:10,1200.00,00:20:00
